In [ ]:
# If needed, install first:
# !pip install pdfplumber pandas openpyxl
 
import os
import re
import pandas as pd
 
try:
    import pdfplumber
except ModuleNotFoundError:
    print("ModuleNotFoundError")
 
def clean_text(text):
    if not text:
        return ""
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text()
            if txt:
                full_text += "\n" + txt
    return clean_text(full_text)
 
 
def cut_unwanted_tail(value):
    if not value:
        return ""
    stop_words = [
        "Medications",
        "Allergies",
        "History",
        "Immunizations",
        "Medication Allergies",
        "Current Medications"
    ]
    pattern = r"\b(" + "|".join(re.escape(w) for w in stop_words) + r")\b.*"
    value = re.sub(pattern, "", value, flags=re.IGNORECASE).strip(" -,:;")
    return clean_text(value)
 
 
def extract_fields(full_text, extraction_headers):
    data = {}
    escaped_headers = [re.escape(h) for h in extraction_headers]

    for header in extraction_headers:
        current = re.escape(header)
        other_headers = escaped_headers.copy()
        other_headers.remove(current)
        next_headers_pattern = "|".join(other_headers)

        pattern = rf"{current}\s*(.*?)\s*(?={next_headers_pattern}|$)"
        match = re.search(pattern, full_text, re.IGNORECASE)

        value = clean_text(match.group(1)) if match else ""

        # Special cleanup for fields that often over-capture
        if header in ["Patient Activity", "Signs & Symptoms", "Chief Complaint", "Secondary Complaint"]:
            value = cut_unwanted_tail(value)

        data[header] = value

    return data


def extract_tables_from_pdf(pdf_path):
    """Extract all tables from PDF, preserving column structure."""
    all_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            all_tables.extend(page.extract_tables() or [])
    return all_tables


def _normalize_label(s):
    """Normalize label for matching (strip, collapse spaces, lowercase)."""
    if not s:
        return ""
    return re.sub(r"\s+", " ", str(s).strip()).lower()


def extract_section_from_tables(all_tables, section_titles, field_headers, section_switches=None):
    """
    Extract key-value pairs from tables where column boundaries are respected.
    Tables have structure: each row contains [Label1, Value1, Label2, Value2, ...].
    Only stores values when the label matches a known header.
    section_titles: list of possible first-row titles (e.g. ["Specialty Patient - CPR"])
    section_switches: optional dict {row_header_substring: output_prefix} for tables with sub-sections.
      e.g. {"Destination Details": "Destination Details - ", "Incident Times": "Incident Times - "}
      When a row's first cell matches, subsequent fields use that prefix until the next switch.
      Fields in field_headers should be the base names (e.g. "Address"); output keys become prefix+field.
    """
    # Find and merge tables for this section
    section_rows = []
    for table in all_tables:
        if not table:
            continue
        first_cell = (table[0][0] if table[0] else None) or ""
        first_cell_norm = _normalize_label(first_cell)
        for title in section_titles:
            if _normalize_label(title) in first_cell_norm or first_cell_norm in _normalize_label(title):
                section_rows.extend(table[1:])  # skip title row
                break

    # Build normalized header -> base header mapping (prefer longer matches)
    base_headers = field_headers if isinstance(field_headers[0], str) else [f[1] for f in field_headers]
    header_map = {}
    for h in sorted(set(base_headers), key=len, reverse=True):
        key = _normalize_label(h)
        if key and key not in header_map:
            header_map[key] = h

    data = {}
    current_prefix = "Incident Details - " if section_switches else ""
    switch_order = list(section_switches.keys()) if section_switches else []

    for row in section_rows:
        if not row:
            continue
        first_cell = _normalize_label(row[0] or "")
        if section_switches:
            for sw in switch_order:
                if _normalize_label(sw) in first_cell or first_cell in _normalize_label(sw):
                    current_prefix = section_switches[sw]
                    break
        for i in range(0, len(row) - 1, 2):
            label = _normalize_label(row[i])
            value = (row[i + 1] or "")
            if isinstance(value, str):
                value = clean_text(value)
            else:
                value = clean_text(str(value))
            if label in header_map:
                base = header_map[label]
                out_key = current_prefix + base if section_switches else base
                data[out_key] = value

    return data


def extract_patient_refusal(all_tables):
    """Extract Patient Refusal section - captures all value cells from the table."""
    for table in all_tables:
        if not table:
            continue
        first_cell = (table[0][0] if table[0] else None) or ""
        if _normalize_label("Patient Refusal") in _normalize_label(first_cell):
            parts = []
            for row in table[1:]:
                if not row:
                    continue
                for i in range(1, len(row), 2):
                    val = row[i] if i < len(row) else None
                    if val and str(val).strip():
                        parts.append(clean_text(str(val)))
            return " ".join(parts).strip() if parts else ""
    return ""


# Excel output names for each table (numbered as per user specification)
OUTPUT_SHEET_NAMES = {
    "patient_info": "1- Patient information",
    "mahi": "2- Medications/allergies/history/immunizations",
    "cpr": "3- Specialty patient - CPR",
    "mvc": "4- Specialty patient - motor vehicle collision",
    "tc": "5- Specialty patient - trauma criteria",
    "cdc": "6- Specialty patient - CDC 2011 Trauma criteria",
    "si": "7- Spinal Immobilization",
    "incident": "8- Incident details/Destination details/ Incident times",
    "insurance": "9- Insurance details",
    "mileage": "10- Mileage/Delays/Additional",
    "ptd": "11- Patient Transport Details",
    "td": "12- Patient Details",
    "pr": "13- Patient Refusal",
}

extraction_headers = [
    "Last", "First", "Middle", "Name Suffix",
    "Sex", "Gender", "DOB", "Age", "Weight", "Height",
    "Pedi Color", "SSN", "Advance Directives", "Resident Status",
    "Patient Resides in Service Area", "Temporary Residence Type",
    "Address", "Address 2", "City", "State", "Zip", "Country",
    "Tel", "Physician", "Phys. Tel", "Ethnicity", "Race",
    "Primary Impression", "Secondary Impression", "Protocols Used",
    "Local Protocol Provided Care Level", "Anatomic Position",
    "Onset Time", "Last Known Well", "Chief Complaint",
    "Secondary Complaint", "Patient's Level of Distress",
    "Signs & Symptoms", "Injury", "Additional Injury",
    "Mechanism of Injury", "Medical/Trauma", "Barriers of Care",
    "Alcohol/Drugs", "Pregnancy", "Initial Patient Acuity",
    "Final Patient Acuity", "Patient Activity"
]
 
final_columns = [
    "Patient Information - Last",
    "Patient Information - First",
    "Patient Information - Middle",
    "Patient Information - Name Suffix",
    "Patient Information - Sex",
    "Patient Information - Gender",
    "Patient Information - DOB",
    "Patient Information - Age",
    "Patient Information - Weight-lbs",
    "Patient Information - Weight-kg",
    "Patient Information - Height-ft",
    "Patient Information - Height-cm",
    "Patient Information - Pedi Color",
    "Patient Information - SSN",
    "Patient Information - Advance Directives",
    "Patient Information - Resident Status",
    "Patient Information - Patient Resides in Service Area",
    "Patient Information - Temporary Residence Type",
    "Patient Information - Address",
    "Patient Information - Address 2",
    "Patient Information - City",
    "Patient Information - State",
    "Patient Information - Zip",
    "Patient Information - Country",
    "Patient Information - Tel",
    "Patient Information - Physician",
    "Patient Information - Phys. Tel",
    "Patient Information - Ethnicity",
    "Patient Information - Race",
    "Clinical Impression - Primary Impression",
    "Clinical Impression - Secondary Impression",
    "Clinical Impression - Protocols Used",
    "Clinical Impression - Local Protocol Provided Care Level",
    "Clinical Impression - Anatomic Position",
    "Clinical Impression - Onset Time",
    "Clinical Impression - Last Known Well",
    "Clinical Impression - Chief Complaint",
    "Clinical Impression - Duration (Chief Complaint)",
    "Clinical Impression - Units (Chief Complaint Duration)",
    "Clinical Impression - Secondary Complaint",
    "Clinical Impression - Duration (Secondary Complaint)",
    "Clinical Impression - Units (Secondary Complaint Duration)",
    "Clinical Impression - Patient's Level of Distress",
    "Clinical Impression - Signs & Symptoms",
    "Clinical Impression - Injury",
    "Clinical Impression - Additional Injury",
    "Clinical Impression - Mechanism of Injury",
    "Clinical Impression - Medical/Trauma",
    "Clinical Impression - Barriers of Care",
    "Clinical Impression - Alcohol/Drugs",
    "Clinical Impression - Pregnancy",
    "Clinical Impression - Initial Patient Acuity",
    "Clinical Impression - Final Patient Acuity",
    "Clinical Impression - Patient Activity"
]
 
pdf_file = "./pdfs/test.pdf"

if not os.path.exists(pdf_file):
    print("File not found. Please check the file name and path.")
else:
    all_tables = extract_tables_from_pdf(pdf_file)
    raw_data = extract_section_from_tables(
        all_tables,
        section_titles=["Patient Information"],
        field_headers=extraction_headers,
    )
 
    output_data = {
        "Patient Information - Last": raw_data.get("Last", ""),
        "Patient Information - First": raw_data.get("First", ""),
        "Patient Information - Middle": raw_data.get("Middle", ""),
        "Patient Information - Name Suffix": raw_data.get("Name Suffix", ""),
        "Patient Information - Sex": raw_data.get("Sex", ""),
        "Patient Information - Gender": raw_data.get("Gender", ""),
        "Patient Information - DOB": raw_data.get("DOB", ""),
        "Patient Information - Age": raw_data.get("Age", ""),
        "Patient Information - Weight-lbs": "",
        "Patient Information - Weight-kg": "",
        "Patient Information - Height-ft": "",
        "Patient Information - Height-cm": "",
        "Patient Information - Pedi Color": raw_data.get("Pedi Color", ""),
        "Patient Information - SSN": raw_data.get("SSN", ""),
        "Patient Information - Advance Directives": raw_data.get("Advance Directives", ""),
        "Patient Information - Resident Status": raw_data.get("Resident Status", ""),
        "Patient Information - Patient Resides in Service Area": raw_data.get("Patient Resides in Service Area", ""),
        "Patient Information - Temporary Residence Type": raw_data.get("Temporary Residence Type", ""),
        "Patient Information - Address": raw_data.get("Address", ""),
        "Patient Information - Address 2": raw_data.get("Address 2", ""),
        "Patient Information - City": raw_data.get("City", ""),
        "Patient Information - State": raw_data.get("State", ""),
        "Patient Information - Zip": raw_data.get("Zip", ""),
        "Patient Information - Country": raw_data.get("Country", ""),
        "Patient Information - Tel": raw_data.get("Tel", ""),
        "Patient Information - Physician": raw_data.get("Physician", ""),
        "Patient Information - Phys. Tel": raw_data.get("Phys. Tel", ""),
        "Patient Information - Ethnicity": raw_data.get("Ethnicity", ""),
        "Patient Information - Race": raw_data.get("Race", ""),
        "Clinical Impression - Primary Impression": raw_data.get("Primary Impression", ""),
        "Clinical Impression - Secondary Impression": raw_data.get("Secondary Impression", ""),
        "Clinical Impression - Protocols Used": raw_data.get("Protocols Used", ""),
        "Clinical Impression - Local Protocol Provided Care Level": raw_data.get("Local Protocol Provided Care Level", ""),
        "Clinical Impression - Anatomic Position": raw_data.get("Anatomic Position", ""),
        "Clinical Impression - Onset Time": raw_data.get("Onset Time", ""),
        "Clinical Impression - Last Known Well": raw_data.get("Last Known Well", ""),
        "Clinical Impression - Chief Complaint": raw_data.get("Chief Complaint", ""),
        "Clinical Impression - Duration (Chief Complaint)": "",
        "Clinical Impression - Units (Chief Complaint Duration)": "",
        "Clinical Impression - Secondary Complaint": raw_data.get("Secondary Complaint", ""),
        "Clinical Impression - Duration (Secondary Complaint)": "",
        "Clinical Impression - Units (Secondary Complaint Duration)": "",
        "Clinical Impression - Patient's Level of Distress": raw_data.get("Patient's Level of Distress", ""),
        "Clinical Impression - Signs & Symptoms": raw_data.get("Signs & Symptoms", ""),
        "Clinical Impression - Injury": raw_data.get("Injury", ""),
        "Clinical Impression - Additional Injury": raw_data.get("Additional Injury", ""),
        "Clinical Impression - Mechanism of Injury": raw_data.get("Mechanism of Injury", ""),
        "Clinical Impression - Medical/Trauma": raw_data.get("Medical/Trauma", ""),
        "Clinical Impression - Barriers of Care": raw_data.get("Barriers of Care", ""),
        "Clinical Impression - Alcohol/Drugs": raw_data.get("Alcohol/Drugs", ""),
        "Clinical Impression - Pregnancy": raw_data.get("Pregnancy", ""),
        "Clinical Impression - Initial Patient Acuity": raw_data.get("Initial Patient Acuity", ""),
        "Clinical Impression - Final Patient Acuity": raw_data.get("Final Patient Acuity", ""),
        "Clinical Impression - Patient Activity": raw_data.get("Patient Activity", "")
    }
 
    weight_text = raw_data.get("Weight", "")
    weight_match = re.search(r"([\d.]+)\s*lbs?\s*-\s*([\d.]+)\s*kg", weight_text, re.IGNORECASE)
    if weight_match:
        output_data["Patient Information - Weight-lbs"] = weight_match.group(1)
        output_data["Patient Information - Weight-kg"] = weight_match.group(2)
    else:
        output_data["Patient Information - Weight-lbs"] = weight_text
 
    height_text = raw_data.get("Height", "")
    height_match = re.search(r"([\d.]+)\s*ft\s*-\s*([\d.]+)\s*cm", height_text, re.IGNORECASE)
    if height_match:
        output_data["Patient Information - Height-ft"] = height_match.group(1)
        output_data["Patient Information - Height-cm"] = height_match.group(2)
    else:
        output_data["Patient Information - Height-ft"] = height_text
 
    df = pd.DataFrame([output_data], columns=final_columns)

    output_dir = os.path.dirname(pdf_file)
    excel_file = os.path.join(output_dir, OUTPUT_SHEET_NAMES["patient_info"] + ".xlsx")
    df.to_excel(excel_file, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 2: Medications / Allergies / History / Immunizations ──────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_mahi = [
    "Medications",
    "Allergies",
    "History",
    "Immunizations",
]

final_columns_mahi = [
    "Medications",
    "Allergies",
    "History",
    "Immunizations",
]

pdf_file_mahi = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_mahi):
    print("File not found. Please check the file name and path.")
else:
    all_tables_mahi = extract_tables_from_pdf(pdf_file_mahi)
    raw_data_mahi = extract_section_from_tables(
        all_tables_mahi,
        section_titles=["Medications/Allergies/History/Immunizations"],
        field_headers=extraction_headers_mahi,
    )

    output_data_mahi = {
        "Medications":   raw_data_mahi.get("Medications", ""),
        "Allergies":     raw_data_mahi.get("Allergies", ""),
        "History":       raw_data_mahi.get("History", ""),
        "Immunizations": raw_data_mahi.get("Immunizations", ""),
    }

    df_mahi = pd.DataFrame([output_data_mahi], columns=final_columns_mahi)

    output_dir_mahi = os.path.dirname(pdf_file_mahi)
    excel_file_mahi = os.path.join(output_dir_mahi, OUTPUT_SHEET_NAMES["mahi"] + ".xlsx")
    df_mahi.to_excel(excel_file_mahi, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 3: Specialty Patient - CPR ────────────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)
# Table-based extraction respects column boundaries (Label|Value pairs per column)

extraction_headers_cpr = [
    "Cardiac Arrest",
    "Cardiac Arrest Etiology",
    "Estimated Time of Arrest",
    "Est Time Collapse to 911",
    "Est Time Collapse to CPR",
    "Arrest Witnessed By",
    "CPR Initiated By",
    "Time 1st CPR",
    "CPR Feedback",
    "ITD Used",
    "Applied AED",
    "Applied By",
    "Defibrillated",
    "CPR Type",
    "Prearrival CPR Instructions",
    "First Defibrillated By",
    "Time of First Defib",
    "Initial ECG Rhythm",
    "Rhythm at Destination",
    "Hypothermia",
    "End of Event",
    "ROSC",
    "ROSC Time",
    "ROSC Occurred",
    "Resuscitation Discontinued",
    "Discontinued Reason",
    "Resuscitation",
    "Expired",
    "Time",
    "Date",
    "Physician",
]

final_columns_cpr = [
    "CPR - Cardiac Arrest",
    "CPR - Cardiac Arrest Etiology",
    "CPR - Estimated Time of Arrest",
    "CPR - Est Time Collapse to 911",
    "CPR - Est Time Collapse to CPR",
    "CPR - Arrest Witnessed By",
    "CPR - CPR Initiated By",
    "CPR - Time 1st CPR",
    "CPR - CPR Feedback",
    "CPR - ITD Used",
    "CPR - Applied AED",
    "CPR - Applied By",
    "CPR - Defibrillated",
    "CPR - CPR Type",
    "CPR - Prearrival CPR Instructions",
    "CPR - First Defibrillated By",
    "CPR - Time of First Defib",
    "CPR - Initial ECG Rhythm",
    "CPR - Rhythm at Destination",
    "CPR - Hypothermia",
    "CPR - End of Event",
    "CPR - ROSC",
    "CPR - ROSC Time",
    "CPR - ROSC Occurred",
    "CPR - Resuscitation Discontinued",
    "CPR - Discontinued Reason",
    "CPR - Resuscitation",
    "In Field Pronouncement - Expired",
    "In Field Pronouncement - Time",
    "In Field Pronouncement - Date",
    "In Field Pronouncement - Physician",
]

pdf_file_cpr = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_cpr):
    print("File not found. Please check the file name and path.")
else:
    all_tables_cpr = extract_tables_from_pdf(pdf_file_cpr)
    raw_data_cpr = extract_section_from_tables(
        all_tables_cpr,
        section_titles=["Specialty Patient - CPR"],
        field_headers=extraction_headers_cpr,
    )

    output_data_cpr = {
        "CPR - Cardiac Arrest":               raw_data_cpr.get("Cardiac Arrest", ""),
        "CPR - Cardiac Arrest Etiology":      raw_data_cpr.get("Cardiac Arrest Etiology", ""),
        "CPR - Estimated Time of Arrest":     raw_data_cpr.get("Estimated Time of Arrest", ""),
        "CPR - Est Time Collapse to 911":     raw_data_cpr.get("Est Time Collapse to 911", ""),
        "CPR - Est Time Collapse to CPR":     raw_data_cpr.get("Est Time Collapse to CPR", ""),
        "CPR - Arrest Witnessed By":          raw_data_cpr.get("Arrest Witnessed By", ""),
        "CPR - CPR Initiated By":             raw_data_cpr.get("CPR Initiated By", ""),
        "CPR - Time 1st CPR":                 raw_data_cpr.get("Time 1st CPR", ""),
        "CPR - CPR Feedback":                 raw_data_cpr.get("CPR Feedback", ""),
        "CPR - ITD Used":                     raw_data_cpr.get("ITD Used", ""),
        "CPR - Applied AED":                  raw_data_cpr.get("Applied AED", ""),
        "CPR - Applied By":                   raw_data_cpr.get("Applied By", ""),
        "CPR - Defibrillated":                raw_data_cpr.get("Defibrillated", ""),
        "CPR - CPR Type":                     raw_data_cpr.get("CPR Type", ""),
        "CPR - Prearrival CPR Instructions":  raw_data_cpr.get("Prearrival CPR Instructions", ""),
        "CPR - First Defibrillated By":       raw_data_cpr.get("First Defibrillated By", ""),
        "CPR - Time of First Defib":          raw_data_cpr.get("Time of First Defib", ""),
        "CPR - Initial ECG Rhythm":           raw_data_cpr.get("Initial ECG Rhythm", ""),
        "CPR - Rhythm at Destination":        raw_data_cpr.get("Rhythm at Destination", ""),
        "CPR - Hypothermia":                  raw_data_cpr.get("Hypothermia", ""),
        "CPR - End of Event":                 raw_data_cpr.get("End of Event", ""),
        "CPR - ROSC":                         raw_data_cpr.get("ROSC", ""),
        "CPR - ROSC Time":                    raw_data_cpr.get("ROSC Time", ""),
        "CPR - ROSC Occurred":                raw_data_cpr.get("ROSC Occurred", ""),
        "CPR - Resuscitation Discontinued":   raw_data_cpr.get("Resuscitation Discontinued", ""),
        "CPR - Discontinued Reason":          raw_data_cpr.get("Discontinued Reason", ""),
        "CPR - Resuscitation":                raw_data_cpr.get("Resuscitation", ""),
        "In Field Pronouncement - Expired":   raw_data_cpr.get("Expired", ""),
        "In Field Pronouncement - Time":      raw_data_cpr.get("Time", ""),
        "In Field Pronouncement - Date":      raw_data_cpr.get("Date", ""),
        "In Field Pronouncement - Physician": raw_data_cpr.get("Physician", ""),
    }

    df_cpr = pd.DataFrame([output_data_cpr], columns=final_columns_cpr)

    output_dir_cpr = os.path.dirname(pdf_file_cpr)
    excel_file_cpr = os.path.join(output_dir_cpr, OUTPUT_SHEET_NAMES["cpr"] + ".xlsx")
    df_cpr.to_excel(excel_file_cpr, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 4: Specialty Patient - Motor Vehicle Collision ────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_mvc = [
    "Patient Injured",
    "Vehicle Type",
    "Position In Vehicle",
    "Seat Row",
    "Number Of Vehicles",
    "Weather",
    "Extrication Required",
    "Estimated Speed",
    "Exterior Damage",
    "Law Enforcement Case #",
    "Collision Indicators",
    "Damage Location",
    "Airbag Deployment",
    "Safety Devices",
    "Extrication Comments",
    "Extrication Time",
]

final_columns_mvc = [
    "Motor Vehicle Collision - Patient Injured",
    "Motor Vehicle Collision - Vehicle Type",
    "Motor Vehicle Collision - Position In Vehicle",
    "Motor Vehicle Collision - Seat Row",
    "Motor Vehicle Collision - Number Of Vehicles",
    "Motor Vehicle Collision - Weather",
    "Motor Vehicle Collision - Extrication Required",
    "Motor Vehicle Collision - Estimated Speed",
    "Motor Vehicle Collision - Exterior Damage",
    "Motor Vehicle Collision - Law Enforcement Case #",
    "Motor Vehicle Collision - Collision Indicators",
    "Motor Vehicle Collision - Damage Location",
    "Motor Vehicle Collision - Airbag Deployment",
    "Motor Vehicle Collision - Safety Devices",
    "Motor Vehicle Collision - Extrication Comments",
    "Motor Vehicle Collision - Extrication Time",
]

pdf_file_mvc = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_mvc):
    print("File not found. Please check the file name and path.")
else:
    all_tables_mvc = extract_tables_from_pdf(pdf_file_mvc)
    raw_data_mvc = extract_section_from_tables(
        all_tables_mvc,
        section_titles=["Specialty Patient - Motor Vehicle Collision"],
        field_headers=extraction_headers_mvc,
    )

    output_data_mvc = {
        "Motor Vehicle Collision - Patient Injured":          raw_data_mvc.get("Patient Injured", ""),
        "Motor Vehicle Collision - Vehicle Type":             raw_data_mvc.get("Vehicle Type", ""),
        "Motor Vehicle Collision - Position In Vehicle":      raw_data_mvc.get("Position In Vehicle", ""),
        "Motor Vehicle Collision - Seat Row":                 raw_data_mvc.get("Seat Row", ""),
        "Motor Vehicle Collision - Number Of Vehicles":       raw_data_mvc.get("Number Of Vehicles", ""),
        "Motor Vehicle Collision - Weather":                  raw_data_mvc.get("Weather", ""),
        "Motor Vehicle Collision - Extrication Required":     raw_data_mvc.get("Extrication Required", ""),
        "Motor Vehicle Collision - Estimated Speed":          raw_data_mvc.get("Estimated Speed", ""),
        "Motor Vehicle Collision - Exterior Damage":          raw_data_mvc.get("Exterior Damage", ""),
        "Motor Vehicle Collision - Law Enforcement Case #":   raw_data_mvc.get("Law Enforcement Case #", ""),
        "Motor Vehicle Collision - Collision Indicators":     raw_data_mvc.get("Collision Indicators", ""),
        "Motor Vehicle Collision - Damage Location":          raw_data_mvc.get("Damage Location", ""),
        "Motor Vehicle Collision - Airbag Deployment":        raw_data_mvc.get("Airbag Deployment", ""),
        "Motor Vehicle Collision - Safety Devices":           raw_data_mvc.get("Safety Devices", ""),
        "Motor Vehicle Collision - Extrication Comments":     raw_data_mvc.get("Extrication Comments", ""),
        "Motor Vehicle Collision - Extrication Time":         raw_data_mvc.get("Extrication Time", ""),
    }

    df_mvc = pd.DataFrame([output_data_mvc], columns=final_columns_mvc)

    output_dir_mvc = os.path.dirname(pdf_file_mvc)
    excel_file_mvc = os.path.join(output_dir_mvc, OUTPUT_SHEET_NAMES["mvc"] + ".xlsx")
    df_mvc.to_excel(excel_file_mvc, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 5: Specialty Patient - Trauma Criteria ────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_tc = [
    "Anatomic",
    "Physiologic",
    "Mechanical",
    "Other Conditions",
    "Trauma Activation",
    "Time",
    "Date",
    "Trauma level",
    "Reason Not Activated",
]

final_columns_tc = [
    "Trauma Criteria - Anatomic",
    "Trauma Criteria - Physiologic",
    "Trauma Criteria - Mechanical",
    "Trauma Criteria - Other Conditions",
    "Trauma Criteria - Trauma Activation",
    "Trauma Criteria - Time",
    "Trauma Criteria - Date",
    "Trauma Criteria - Trauma level",
    "Trauma Criteria - Reason Not Activated",
]

pdf_file_tc = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_tc):
    print("File not found. Please check the file name and path.")
else:
    all_tables_tc = extract_tables_from_pdf(pdf_file_tc)
    raw_data_tc = extract_section_from_tables(
        all_tables_tc,
        section_titles=["Specialty Patient - Trauma Criteria"],
        field_headers=extraction_headers_tc,
    )

    output_data_tc = {
        "Trauma Criteria - Anatomic":            raw_data_tc.get("Anatomic", ""),
        "Trauma Criteria - Physiologic":         raw_data_tc.get("Physiologic", ""),
        "Trauma Criteria - Mechanical":          raw_data_tc.get("Mechanical", ""),
        "Trauma Criteria - Other Conditions":    raw_data_tc.get("Other Conditions", ""),
        "Trauma Criteria - Trauma Activation":   raw_data_tc.get("Trauma Activation", ""),
        "Trauma Criteria - Time":                raw_data_tc.get("Time", ""),
        "Trauma Criteria - Date":                raw_data_tc.get("Date", ""),
        "Trauma Criteria - Trauma level":        raw_data_tc.get("Trauma level", ""),
        "Trauma Criteria - Reason Not Activated": raw_data_tc.get("Reason Not Activated", ""),
    }

    df_tc = pd.DataFrame([output_data_tc], columns=final_columns_tc)

    output_dir_tc = os.path.dirname(pdf_file_tc)
    excel_file_tc = os.path.join(output_dir_tc, OUTPUT_SHEET_NAMES["tc"] + ".xlsx")
    df_tc.to_excel(excel_file_tc, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 6: Specialty Patient - CDC 2011 Trauma Criteria ───────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_cdc = [
    "Vital Signs",
    "Anatomy of Injury",
    "Mechanism of Injury",
    "Special Considerations",
    "Trauma Activation",
    "Time",
    "Date",
    "level",
    "Reason Not Activated",
]

final_columns_cdc = [
    "CDC 2011 Trauma Criteria - Vital Signs",
    "CDC 2011 Trauma Criteria - Anatomy of Injury",
    "CDC 2011 Trauma Criteria - Mechanism of Injury",
    "CDC 2011 Trauma Criteria - Special Considerations",
    "CDC 2011 Trauma Criteria - Trauma Activation",
    "CDC 2011 Trauma Criteria - Time",
    "CDC 2011 Trauma Criteria - Date",
    "CDC 2011 Trauma Criteria - level",
    "CDC 2011 Trauma Criteria - Reason Not Activated",
]

pdf_file_cdc = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_cdc):
    print("File not found. Please check the file name and path.")
else:
    all_tables_cdc = extract_tables_from_pdf(pdf_file_cdc)
    raw_data_cdc = extract_section_from_tables(
        all_tables_cdc,
        section_titles=["Specialty Patient - CDC 2011 Trauma Criteria"],
        field_headers=extraction_headers_cdc,
    )

    output_data_cdc = {
        "CDC 2011 Trauma Criteria - Vital Signs":            raw_data_cdc.get("Vital Signs", ""),
        "CDC 2011 Trauma Criteria - Anatomy of Injury":      raw_data_cdc.get("Anatomy of Injury", ""),
        "CDC 2011 Trauma Criteria - Mechanism of Injury":    raw_data_cdc.get("Mechanism of Injury", ""),
        "CDC 2011 Trauma Criteria - Special Considerations": raw_data_cdc.get("Special Considerations", ""),
        "CDC 2011 Trauma Criteria - Trauma Activation":      raw_data_cdc.get("Trauma Activation", ""),
        "CDC 2011 Trauma Criteria - Time":                   raw_data_cdc.get("Time", ""),
        "CDC 2011 Trauma Criteria - Date":                   raw_data_cdc.get("Date", ""),
        "CDC 2011 Trauma Criteria - level":                  raw_data_cdc.get("level", ""),
        "CDC 2011 Trauma Criteria - Reason Not Activated":   raw_data_cdc.get("Reason Not Activated", ""),
    }

    df_cdc = pd.DataFrame([output_data_cdc], columns=final_columns_cdc)

    output_dir_cdc = os.path.dirname(pdf_file_cdc)
    excel_file_cdc = os.path.join(output_dir_cdc, OUTPUT_SHEET_NAMES["cdc"] + ".xlsx")
    df_cdc.to_excel(excel_file_cdc, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 7: Spinal Immobilization ──────────────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_si = [
    "Immobilization Recommended?",
    "Altered Mental Status",
    "Evidence of Alcohol/Drug Impairment",
    "Distracting Injury",
    "Neurologic Deficit",
    "Spinal Pain/Tenderness",
]

final_columns_si = [
    "Spinal Immobilization - Immobilization Recommended?",
    "Spinal Immobilization - Altered Mental Status",
    "Spinal Immobilization - Evidence of Alcohol/Drug Impairment",
    "Spinal Immobilization - Distracting Injury",
    "Spinal Immobilization - Neurologic Deficit",
    "Spinal Immobilization - Spinal Pain/Tenderness",
]

pdf_file_si = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_si):
    print("File not found. Please check the file name and path.")
else:
    all_tables_si = extract_tables_from_pdf(pdf_file_si)
    raw_data_si = extract_section_from_tables(
        all_tables_si,
        section_titles=["Spinal Immobilization", "Specialty Patient - Spinal Immobilization"],
        field_headers=extraction_headers_si,
    )

    output_data_si = {
        "Spinal Immobilization - Immobilization Recommended?":        raw_data_si.get("Immobilization Recommended?", ""),
        "Spinal Immobilization - Altered Mental Status":               raw_data_si.get("Altered Mental Status", ""),
        "Spinal Immobilization - Evidence of Alcohol/Drug Impairment": raw_data_si.get("Evidence of Alcohol/Drug Impairment", ""),
        "Spinal Immobilization - Distracting Injury":                  raw_data_si.get("Distracting Injury", ""),
        "Spinal Immobilization - Neurologic Deficit":                  raw_data_si.get("Neurologic Deficit", ""),
        "Spinal Immobilization - Spinal Pain/Tenderness":              raw_data_si.get("Spinal Pain/Tenderness", ""),
    }

    df_si = pd.DataFrame([output_data_si], columns=final_columns_si)

    output_dir_si = os.path.dirname(pdf_file_si)
    excel_file_si = os.path.join(output_dir_si, OUTPUT_SHEET_NAMES["si"] + ".xlsx")
    df_si.to_excel(excel_file_si, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 8: Incident Details / Destination Details / Incident Times ─────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)
# Uses section_switches to separate Incident Details, Destination Details, Incident Times

extraction_headers_id = [
    "Location Type",
    "Location",
    "Address",
    "Address 2",
    "Mile Marker",
    "City",
    "County",
    "State",
    "Zip",
    "Country",
    "Medic Unit",
    "Medic Vehicle",
    "Run Type",
    "Response Mode",
    "Response Mode Descriptors",
    "Shift",
    "Zone",
    "Level of Service",
    "EMD Complaint",
    "EMD Card Number",
    "Dispatch Priority",
]

extraction_headers_dd = [
    "Disposition",
    "Unit Disposition",
    "Patient Evaluation and/or Care Disposition",
    "Crew Disposition",
    "Transport Disposition",
    "Transport Mode",
    "Reason for Refusal or Release",
    "Transport Mode Descriptors",
    "Transport Due To",
    "Transported To",
    "Requested By",
    "Transferred To",
    "Transferred Unit",
    "Destination",
    "Department",
    "Address",
    "Address 2",
    "City",
    "County",
    "State",
    "Zip",
    "Country",
    "Zone",
    "Condition at Destination",
    "State Wristband #",
    "Destination Record #",
    "Trauma Registry ID",
    "STEMI Registry ID",
    "Stroke Registry ID",
]

extraction_headers_it = [
    "PSAP Call",
    "Dispatch Notified",
    "Call Received",
    "Dispatched",
    "En Route",
    "Staged",
    "Resp on Scene",
    "On Scene",
    "At Patient",
    "Care Transferred",
    "Depart Scene",
    "At Destination",
    "Pt. Transferred",
    "Call Closed",
    "In District",
    "At Landing Area",
]

final_columns_incident = [
    "Incident Details - Location Type",
    "Incident Details - Location",
    "Incident Details - Address",
    "Incident Details - Address 2",
    "Incident Details - Mile Marker",
    "Incident Details - City",
    "Incident Details - County",
    "Incident Details - State",
    "Incident Details - Zip",
    "Incident Details - Country",
    "Incident Details - Medic Unit",
    "Incident Details - Medic Vehicle",
    "Incident Details - Run Type",
    "Incident Details - Response Mode",
    "Incident Details - Response Mode Descriptors",
    "Incident Details - Shift",
    "Incident Details - Zone",
    "Incident Details - Level of Service",
    "Incident Details - EMD Complaint",
    "Incident Details - EMD Card Number",
    "Incident Details - Dispatch Priority",
    "Destination Details - Disposition",
    "Destination Details - Unit Disposition",
    "Destination Details - Patient Evaluation and/or Care Disposition",
    "Destination Details - Crew Disposition",
    "Destination Details - Transport Disposition",
    "Destination Details - Transport Mode",
    "Destination Details - Reason for Refusal or Release",
    "Destination Details - Transport Mode Descriptors",
    "Destination Details - Transport Due To",
    "Destination Details - Transported To",
    "Destination Details - Requested By",
    "Destination Details - Transferred To",
    "Destination Details - Transferred Unit",
    "Destination Details - Destination",
    "Destination Details - Department",
    "Destination Details - Address",
    "Destination Details - Address 2",
    "Destination Details - City",
    "Destination Details - County",
    "Destination Details - State",
    "Destination Details - Zip",
    "Destination Details - Country",
    "Destination Details - Zone",
    "Destination Details - Condition at Destination",
    "Destination Details - State Wristband #",
    "Destination Details - Destination Record #",
    "Destination Details - Trauma Registry ID",
    "Destination Details - STEMI Registry ID",
    "Destination Details - Stroke Registry ID",
    "Incident Times - PSAP Call",
    "Incident Times - Dispatch Notified",
    "Incident Times - Call Received",
    "Incident Times - Dispatched",
    "Incident Times - En Route",
    "Incident Times - Staged",
    "Incident Times - Resp on Scene",
    "Incident Times - On Scene",
    "Incident Times - At Patient",
    "Incident Times - Care Transferred",
    "Incident Times - Depart Scene",
    "Incident Times - At Destination",
    "Incident Times - Pt. Transferred",
    "Incident Times - Call Closed",
    "Incident Times - In District",
    "Incident Times - At Landing Area",
]

pdf_file_incident = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_incident):
    print("File not found. Please check the file name and path.")
else:
    all_tables_incident = extract_tables_from_pdf(pdf_file_incident)
    all_headers_incident = extraction_headers_id + extraction_headers_dd + extraction_headers_it
    raw_data_incident = extract_section_from_tables(
        all_tables_incident,
        section_titles=["Incident Details"],
        field_headers=all_headers_incident,
        section_switches={
            "Destination Details": "Destination Details - ",
            "Incident Times": "Incident Times - ",
        },
    )

    output_data_incident = {
        "Incident Details - Location Type":                              raw_data_incident.get("Incident Details - Location Type", ""),
        "Incident Details - Location":                                   raw_data_incident.get("Incident Details - Location", ""),
        "Incident Details - Address":                                    raw_data_incident.get("Incident Details - Address", ""),
        "Incident Details - Address 2":                                  raw_data_incident.get("Incident Details - Address 2", ""),
        "Incident Details - Mile Marker":                                raw_data_incident.get("Incident Details - Mile Marker", ""),
        "Incident Details - City":                                       raw_data_incident.get("Incident Details - City", ""),
        "Incident Details - County":                                     raw_data_incident.get("Incident Details - County", ""),
        "Incident Details - State":                                      raw_data_incident.get("Incident Details - State", ""),
        "Incident Details - Zip":                                        raw_data_incident.get("Incident Details - Zip", ""),
        "Incident Details - Country":                                    raw_data_incident.get("Incident Details - Country", ""),
        "Incident Details - Medic Unit":                                 raw_data_incident.get("Incident Details - Medic Unit", ""),
        "Incident Details - Medic Vehicle":                              raw_data_incident.get("Incident Details - Medic Vehicle", ""),
        "Incident Details - Run Type":                                   raw_data_incident.get("Incident Details - Run Type", ""),
        "Incident Details - Response Mode":                              raw_data_incident.get("Incident Details - Response Mode", ""),
        "Incident Details - Response Mode Descriptors":                  raw_data_incident.get("Incident Details - Response Mode Descriptors", ""),
        "Incident Details - Shift":                                      raw_data_incident.get("Incident Details - Shift", ""),
        "Incident Details - Zone":                                       raw_data_incident.get("Incident Details - Zone", ""),
        "Incident Details - Level of Service":                           raw_data_incident.get("Incident Details - Level of Service", ""),
        "Incident Details - EMD Complaint":                              raw_data_incident.get("Incident Details - EMD Complaint", ""),
        "Incident Details - EMD Card Number":                            raw_data_incident.get("Incident Details - EMD Card Number", ""),
        "Incident Details - Dispatch Priority":                          raw_data_incident.get("Incident Details - Dispatch Priority", ""),
        "Destination Details - Disposition":                             raw_data_incident.get("Destination Details - Disposition", ""),
        "Destination Details - Unit Disposition":                        raw_data_incident.get("Destination Details - Unit Disposition", ""),
        "Destination Details - Patient Evaluation and/or Care Disposition": raw_data_incident.get("Destination Details - Patient Evaluation and/or Care Disposition", ""),
        "Destination Details - Crew Disposition":                        raw_data_incident.get("Destination Details - Crew Disposition", ""),
        "Destination Details - Transport Disposition":                   raw_data_incident.get("Destination Details - Transport Disposition", ""),
        "Destination Details - Transport Mode":                          raw_data_incident.get("Destination Details - Transport Mode", ""),
        "Destination Details - Reason for Refusal or Release":           raw_data_incident.get("Destination Details - Reason for Refusal or Release", ""),
        "Destination Details - Transport Mode Descriptors":              raw_data_incident.get("Destination Details - Transport Mode Descriptors", ""),
        "Destination Details - Transport Due To":                        raw_data_incident.get("Destination Details - Transport Due To", ""),
        "Destination Details - Transported To":                          raw_data_incident.get("Destination Details - Transported To", ""),
        "Destination Details - Requested By":                            raw_data_incident.get("Destination Details - Requested By", ""),
        "Destination Details - Transferred To":                          raw_data_incident.get("Destination Details - Transferred To", ""),
        "Destination Details - Transferred Unit":                        raw_data_incident.get("Destination Details - Transferred Unit", ""),
        "Destination Details - Destination":                             raw_data_incident.get("Destination Details - Destination", ""),
        "Destination Details - Department":                              raw_data_incident.get("Destination Details - Department", ""),
        "Destination Details - Address":                                 raw_data_incident.get("Destination Details - Address", ""),
        "Destination Details - Address 2":                               raw_data_incident.get("Destination Details - Address 2", ""),
        "Destination Details - City":                                    raw_data_incident.get("Destination Details - City", ""),
        "Destination Details - County":                                  raw_data_incident.get("Destination Details - County", ""),
        "Destination Details - State":                                   raw_data_incident.get("Destination Details - State", ""),
        "Destination Details - Zip":                                     raw_data_incident.get("Destination Details - Zip", ""),
        "Destination Details - Country":                                 raw_data_incident.get("Destination Details - Country", ""),
        "Destination Details - Zone":                                    raw_data_incident.get("Destination Details - Zone", ""),
        "Destination Details - Condition at Destination":                raw_data_incident.get("Destination Details - Condition at Destination", ""),
        "Destination Details - State Wristband #":                       raw_data_incident.get("Destination Details - State Wristband #", ""),
        "Destination Details - Destination Record #":                    raw_data_incident.get("Destination Details - Destination Record #", ""),
        "Destination Details - Trauma Registry ID":                      raw_data_incident.get("Destination Details - Trauma Registry ID", ""),
        "Destination Details - STEMI Registry ID":                       raw_data_incident.get("Destination Details - STEMI Registry ID", ""),
        "Destination Details - Stroke Registry ID":                      raw_data_incident.get("Destination Details - Stroke Registry ID", ""),
        "Incident Times - PSAP Call":                                    raw_data_incident.get("Incident Times - PSAP Call", ""),
        "Incident Times - Dispatch Notified":                            raw_data_incident.get("Incident Times - Dispatch Notified", ""),
        "Incident Times - Call Received":                                raw_data_incident.get("Incident Times - Call Received", ""),
        "Incident Times - Dispatched":                                   raw_data_incident.get("Incident Times - Dispatched", ""),
        "Incident Times - En Route":                                     raw_data_incident.get("Incident Times - En Route", ""),
        "Incident Times - Staged":                                       raw_data_incident.get("Incident Times - Staged", ""),
        "Incident Times - Resp on Scene":                                raw_data_incident.get("Incident Times - Resp on Scene", ""),
        "Incident Times - On Scene":                                     raw_data_incident.get("Incident Times - On Scene", ""),
        "Incident Times - At Patient":                                   raw_data_incident.get("Incident Times - At Patient", ""),
        "Incident Times - Care Transferred":                             raw_data_incident.get("Incident Times - Care Transferred", ""),
        "Incident Times - Depart Scene":                                 raw_data_incident.get("Incident Times - Depart Scene", ""),
        "Incident Times - At Destination":                               raw_data_incident.get("Incident Times - At Destination", ""),
        "Incident Times - Pt. Transferred":                              raw_data_incident.get("Incident Times - Pt. Transferred", ""),
        "Incident Times - Call Closed":                                  raw_data_incident.get("Incident Times - Call Closed", ""),
        "Incident Times - In District":                                  raw_data_incident.get("Incident Times - In District", ""),
        "Incident Times - At Landing Area":                              raw_data_incident.get("Incident Times - At Landing Area", ""),
    }

    df_incident = pd.DataFrame([output_data_incident], columns=final_columns_incident)

    output_dir_incident = os.path.dirname(pdf_file_incident)
    excel_file_incident = os.path.join(output_dir_incident, OUTPUT_SHEET_NAMES["incident"] + ".xlsx")
    df_incident.to_excel(excel_file_incident, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 9: Insurance Details ───────────────────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_ins = [
    "Insured's Name",
    "Relationship",
    "Insured SSN",
    "Insured DOB",
    "Address1",
    "Address2",
    "Address3",
    "City",
    "State",
    "Zip",
    "Country",
    "Primary Payer",
    "Medicare",
    "Medicaid",
    "Primary Insurance",
    "Primary Insurance Policy #",
    "Primary Insurance Group Name",
    "Primary Insurance Group #",
    "Secondary Ins",
    "Secondary Insurance Policy #",
    "Secondary Insurance Group Name",
    "Secondary Insurance Group #",
    "Dispatch Nature",
    "Response Urgency",
    "Job Related Injury",
    "Employer",
    "Contact",
    "Phone",
    "Mileage to Closest Hospital",
]

final_columns_ins = [
    "Insured's Name",
    "Relationship",
    "Insured SSN",
    "Insured DOB",
    "Address1",
    "Address2",
    "Address3",
    "City",
    "State",
    "Zip",
    "Country",
    "Primary Payer",
    "Medicare",
    "Medicaid",
    "Primary Insurance",
    "Primary Insurance Policy #",
    "Primary Insurance Group Name",
    "Primary Insurance Group #",
    "Secondary Ins",
    "Secondary Insurance Policy #",
    "Secondary Insurance Group Name",
    "Secondary Insurance Group #",
    "Dispatch Nature",
    "Response Urgency",
    "Job Related Injury",
    "Employer",
    "Contact",
    "Phone",
    "Mileage to Closest Hospital",
]

pdf_file_ins = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_ins):
    print("File not found. Please check the file name and path.")
else:
    all_tables_ins = extract_tables_from_pdf(pdf_file_ins)
    raw_data_ins = extract_section_from_tables(
        all_tables_ins,
        section_titles=["Insurance Details"],
        field_headers=extraction_headers_ins,
    )

    output_data_ins = {
        "Insured's Name":                  raw_data_ins.get("Insured's Name", ""),
        "Relationship":                    raw_data_ins.get("Relationship", ""),
        "Insured SSN":                     raw_data_ins.get("Insured SSN", ""),
        "Insured DOB":                     raw_data_ins.get("Insured DOB", ""),
        "Address1":                        raw_data_ins.get("Address1", ""),
        "Address2":                        raw_data_ins.get("Address2", ""),
        "Address3":                        raw_data_ins.get("Address3", ""),
        "City":                            raw_data_ins.get("City", ""),
        "State":                           raw_data_ins.get("State", ""),
        "Zip":                             raw_data_ins.get("Zip", ""),
        "Country":                         raw_data_ins.get("Country", ""),
        "Primary Payer":                   raw_data_ins.get("Primary Payer", ""),
        "Medicare":                        raw_data_ins.get("Medicare", ""),
        "Medicaid":                        raw_data_ins.get("Medicaid", ""),
        "Primary Insurance":               raw_data_ins.get("Primary Insurance", ""),
        "Primary Insurance Policy #":      raw_data_ins.get("Primary Insurance Policy #", ""),
        "Primary Insurance Group Name":    raw_data_ins.get("Primary Insurance Group Name", ""),
        "Primary Insurance Group #":       raw_data_ins.get("Primary Insurance Group #", ""),
        "Secondary Ins":                   raw_data_ins.get("Secondary Ins", ""),
        "Secondary Insurance Policy #":    raw_data_ins.get("Secondary Insurance Policy #", ""),
        "Secondary Insurance Group Name":  raw_data_ins.get("Secondary Insurance Group Name", ""),
        "Secondary Insurance Group #":     raw_data_ins.get("Secondary Insurance Group #", ""),
        "Dispatch Nature":                 raw_data_ins.get("Dispatch Nature", ""),
        "Response Urgency":                raw_data_ins.get("Response Urgency", ""),
        "Job Related Injury":              raw_data_ins.get("Job Related Injury", ""),
        "Employer":                        raw_data_ins.get("Employer", ""),
        "Contact":                         raw_data_ins.get("Contact", ""),
        "Phone":                           raw_data_ins.get("Phone", ""),
        "Mileage to Closest Hospital":     raw_data_ins.get("Mileage to Closest Hospital", ""),
    }

    df_ins = pd.DataFrame([output_data_ins], columns=final_columns_ins)

    output_dir_ins = os.path.dirname(pdf_file_ins)
    excel_file_ins = os.path.join(output_dir_ins, OUTPUT_SHEET_NAMES["insurance"] + ".xlsx")
    df_ins.to_excel(excel_file_ins, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 10: Mileage / Delays / Additional ──────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_mil = [
    "Scene",
    "Destination",
    "Loaded Miles",
    "Start",
    "End",
    "Total Miles",
    "Dispatch Delays",
    "Response Delays",
    "Scene Delays",
    "Transport Delays",
    "Turn Around Delays",
    "Additional Agencies",
]

final_columns_mil = [
    "Mileage - Scene",
    "Mileage - Destination",
    "Mileage - Loaded Miles",
    "Mileage - Start",
    "Mileage - End",
    "Mileage - Total Miles",
    "Dispatch Delays",
    "Response Delays",
    "Scene Delays",
    "Transport Delays",
    "Turn Around Delays",
    "Additional Agencies",
]

pdf_file_mil = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_mil):
    print("File not found. Please check the file name and path.")
else:
    all_tables_mil = extract_tables_from_pdf(pdf_file_mil)
    raw_data_mil = extract_section_from_tables(
        all_tables_mil,
        section_titles=["Mileage"],
        field_headers=extraction_headers_mil,
    )

    output_data_mil = {
        "Mileage - Scene":       raw_data_mil.get("Scene", ""),
        "Mileage - Destination": raw_data_mil.get("Destination", ""),
        "Mileage - Loaded Miles": raw_data_mil.get("Loaded Miles", ""),
        "Mileage - Start":       raw_data_mil.get("Start", ""),
        "Mileage - End":         raw_data_mil.get("End", ""),
        "Mileage - Total Miles": raw_data_mil.get("Total Miles", ""),
        "Dispatch Delays":       raw_data_mil.get("Dispatch Delays", ""),
        "Response Delays":       raw_data_mil.get("Response Delays", ""),
        "Scene Delays":          raw_data_mil.get("Scene Delays", ""),
        "Transport Delays":      raw_data_mil.get("Transport Delays", ""),
        "Turn Around Delays":    raw_data_mil.get("Turn Around Delays", ""),
        "Additional Agencies":   raw_data_mil.get("Additional Agencies", ""),
    }

    df_mil = pd.DataFrame([output_data_mil], columns=final_columns_mil)

    output_dir_mil = os.path.dirname(pdf_file_mil)
    excel_file_mil = os.path.join(output_dir_mil, OUTPUT_SHEET_NAMES["mileage"] + ".xlsx")
    df_mil.to_excel(excel_file_mil, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 11: Patient Transport Details ─────────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_ptd = [
    "How was Patient Moved To Stretcher",
    "How was Patient Moved From Ambulance",
    "Condition of Patient at Destination",
    "How was Patient Moved To Ambulance",
    "Patient Position During Transport",
]

final_columns_ptd = [
    "Patient Transport Details - How was Patient Moved To Stretcher",
    "Patient Transport Details - How was Patient Moved From Ambulance",
    "Patient Transport Details - Condition of Patient at Destination",
    "Patient Transport Details - How was Patient Moved To Ambulance",
    "Patient Transport Details - Patient Position During Transport",
]

pdf_file_ptd = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_ptd):
    print("File not found. Please check the file name and path.")
else:
    all_tables_ptd = extract_tables_from_pdf(pdf_file_ptd)
    raw_data_ptd = extract_section_from_tables(
        all_tables_ptd,
        section_titles=["Patient Transport Details"],
        field_headers=extraction_headers_ptd,
    )

    output_data_ptd = {
        "Patient Transport Details - How was Patient Moved To Stretcher":  raw_data_ptd.get("How was Patient Moved To Stretcher", ""),
        "Patient Transport Details - How was Patient Moved From Ambulance": raw_data_ptd.get("How was Patient Moved From Ambulance", ""),
        "Patient Transport Details - Condition of Patient at Destination":  raw_data_ptd.get("Condition of Patient at Destination", ""),
        "Patient Transport Details - How was Patient Moved To Ambulance":   raw_data_ptd.get("How was Patient Moved To Ambulance", ""),
        "Patient Transport Details - Patient Position During Transport":    raw_data_ptd.get("Patient Position During Transport", ""),
    }

    df_ptd = pd.DataFrame([output_data_ptd], columns=final_columns_ptd)

    output_dir_ptd = os.path.dirname(pdf_file_ptd)
    excel_file_ptd = os.path.join(output_dir_ptd, OUTPUT_SHEET_NAMES["ptd"] + ".xlsx")
    df_ptd.to_excel(excel_file_ptd, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 12: Transfer Details (Patient Details) ─────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_section_from_tables  (defined in Cell 0)

extraction_headers_td = [
    "PAN",
    "Prior Authorization Code Payer",
    "PCS",
    "Interfacility Transfer or Medical Transport Reason",
    "ABN",
    "CMS Service Level",
    ">ICD-9 Code",
    "Transport Assessment",
    "Specialty Care Transport Provider",
    "Transfer Reason",
    "Justification for Transfer",
    "Other/Services",
    "Medical Necessity",
    "Sending Physician",
    "Sending Record #",
    "Receiving Physician",
    "Condition Code",
    "Condition Code Modifiers",
]

final_columns_td = [
    "Transfer Details - PAN",
    "Transfer Details - Prior Authorization Code Payer",
    "Transfer Details - PCS",
    "Transfer Details - Interfacility Transfer or Medical Transport Reason",
    "Transfer Details - ABN",
    "Transfer Details - CMS Service Level",
    "Transfer Details - >ICD-9 Code",
    "Transfer Details - Transport Assessment",
    "Transfer Details - Specialty Care Transport Provider",
    "Transfer Details - Transfer Reason",
    "Transfer Details - Justification for Transfer",
    "Transfer Details - Other/Services",
    "Transfer Details - Medical Necessity",
    "Transfer Details - Sending Physician",
    "Transfer Details - Sending Record #",
    "Transfer Details - Receiving Physician",
    "Transfer Details - Condition Code",
    "Transfer Details - Condition Code Modifiers",
]

pdf_file_td = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_td):
    print("File not found. Please check the file name and path.")
else:
    all_tables_td = extract_tables_from_pdf(pdf_file_td)
    raw_data_td = extract_section_from_tables(
        all_tables_td,
        section_titles=["Transfer Details"],
        field_headers=extraction_headers_td,
    )

    output_data_td = {
        "Transfer Details - PAN":                                            raw_data_td.get("PAN", ""),
        "Transfer Details - Prior Authorization Code Payer":                 raw_data_td.get("Prior Authorization Code Payer", ""),
        "Transfer Details - PCS":                                            raw_data_td.get("PCS", ""),
        "Transfer Details - Interfacility Transfer or Medical Transport Reason": raw_data_td.get("Interfacility Transfer or Medical Transport Reason", ""),
        "Transfer Details - ABN":                                            raw_data_td.get("ABN", ""),
        "Transfer Details - CMS Service Level":                              raw_data_td.get("CMS Service Level", ""),
        "Transfer Details - >ICD-9 Code":                                    raw_data_td.get(">ICD-9 Code", ""),
        "Transfer Details - Transport Assessment":                           raw_data_td.get("Transport Assessment", ""),
        "Transfer Details - Specialty Care Transport Provider":              raw_data_td.get("Specialty Care Transport Provider", ""),
        "Transfer Details - Transfer Reason":                                raw_data_td.get("Transfer Reason", ""),
        "Transfer Details - Justification for Transfer":                     raw_data_td.get("Justification for Transfer", ""),
        "Transfer Details - Other/Services":                                 raw_data_td.get("Other/Services", ""),
        "Transfer Details - Medical Necessity":                              raw_data_td.get("Medical Necessity", ""),
        "Transfer Details - Sending Physician":                              raw_data_td.get("Sending Physician", ""),
        "Transfer Details - Sending Record #":                               raw_data_td.get("Sending Record #", ""),
        "Transfer Details - Receiving Physician":                            raw_data_td.get("Receiving Physician", ""),
        "Transfer Details - Condition Code":                                 raw_data_td.get("Condition Code", ""),
        "Transfer Details - Condition Code Modifiers":                       raw_data_td.get("Condition Code Modifiers", ""),
    }

    df_td = pd.DataFrame([output_data_td], columns=final_columns_td)

    output_dir_td = os.path.dirname(pdf_file_td)
    excel_file_td = os.path.join(output_dir_td, OUTPUT_SHEET_NAMES["td"] + ".xlsx")
    df_td.to_excel(excel_file_td, index=False)
    print("Excel successfully generated")

In [ ]:
# ── TABLE 13: Patient Refusal ────────────────────────────────────────────────
# Reuses: clean_text, extract_tables_from_pdf, extract_patient_refusal  (defined in Cell 1)

final_columns_pr = [
    "Patient Refusal",
]

pdf_file_pr = "./pdfs/test.pdf"

if not os.path.exists(pdf_file_pr):
    print("File not found. Please check the file name and path.")
else:
    all_tables_pr = extract_tables_from_pdf(pdf_file_pr)
    pr_value = extract_patient_refusal(all_tables_pr)
    if not pr_value:
        full_text_pr = extract_text_from_pdf(pdf_file_pr)
        pr_match = re.search(r"Patient Refusal\s*(.*?)\s*$", full_text_pr, re.IGNORECASE)
        pr_value = clean_text(pr_match.group(1)) if pr_match else ""

    output_data_pr = {
        "Patient Refusal": pr_value,
    }

    df_pr = pd.DataFrame([output_data_pr], columns=final_columns_pr)

    output_dir_pr = os.path.dirname(pdf_file_pr)
    excel_file_pr = os.path.join(output_dir_pr, OUTPUT_SHEET_NAMES["pr"] + ".xlsx")
    df_pr.to_excel(excel_file_pr, index=False)
    print("Excel successfully generated")